In [0]:
df = spark.table('gizmobox.bronze.py_orders')
display(df)

In [0]:
df.printSchema()

In [0]:
parsed_df = spark.sql("""
SELECT
  from_json(value,
    'STRUCT<
        order_id: INT,
        customer_id: LONG,
        order_date: DATE,
        transaction_timestamp: TIMESTAMP,
        total_amount: INT,
        payment_method: STRING,
        order_status: STRING,
        items: ARRAY<
            STRUCT<
                item_id: INT,
                name: STRING,
                category: STRING,
                price: INT,
                quantity: INT,
                details: STRUCT<brand: STRING, color: STRING>
            >
        >
    >'
  ) AS data
FROM gizmobox.bronze.py_orders
""")

In [0]:
display(parsed_df)

In [0]:
final_df = parsed_df.select("data.*")
display(final_df)

In [0]:
from pyspark.sql.functions import *
final_df = final_df.withColumn(
    "item",
    explode("items")
)

In [0]:
display(final_df)

In [0]:
final_df = final_df.select(
    "order_id",
    "customer_id",
    "order_date",
    "transaction_timestamp",
    "total_amount",
    "payment_method",
    "order_status",
    col("item.item_id").alias("item_id"),
    col("item.name").alias("item_name"),
    col("item.category").alias("category"),
    col("item.price").alias("price"),
    col("item.quantity").alias("quantity"),
    col("item.details.brand").alias("brand"),
    col("item.details.color").alias("color")
)
display(final_df)
    

In [0]:
#final_df = final_df.dropDuplicates()
display(final_df.filter(col('customer_id') == '2344'))

In [0]:
# writing df to silver
final_df.writeTo("gizmobox.silver.py_orders").createOrReplace()